# FIP 00 — FIP and motion energy: exploration

How do NAc DA, PL LC-axon calcium and NAc ACh relate to gross movement (motion energy)?

The notebook works through one session first and then repeats the measures over every curated session with motion energy. For one session it plots traces and go-cue averages, aligns each signal to motion-energy onsets and motion energy to each signal's transients, and cross-correlates them. Across sessions it repeats the two alignments, compares the level within the trial with the level in the inter-trial interval, and splits go-cue responses by reward/failure streak.

FIP data are `DA_NE_4channels` (asset `6babbf3d-6970-4456-aab5-d730ed57c269`), with the JSON curation applied at load. Motion energy comes from the aligned ME table (`fip_motion_energy_aligned`, computed by `aind-motion-energy`). `df_fip` holds one trace per `event`, with timestamps in seconds from the first go cue.

## Imports & setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Shared fip_* setup (loading, curation, session select, example channels, motion energy on the
# FIP clock) is fip_utils; generic signal helpers (zscore / threshold_onsets / norm_xcorr) are
# signal_utils.
# The same module backs fip_01_movement_value_coding.ipynb and fip_02_ne_only_events.ipynb.
# autoreload so editing fip_utils.py doesn't cost a kernel restart + a full session reload.
%load_ext autoreload
%autoreload 2
import fip_utils as fu
import signal_utils as su
import stats_utils as st
import plot_utils as pu

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)


## Data loading and curation

`fu.load_curated_sessions` reads the parquet hierarchy (`load_nwb_list`) and applies the JSON curation (`apply_curation_nwb_list`): region label per fiber, bad channels and sessions dropped. The `..._firstpass` curation file holds this asset's fiber-to-region map.

In [ ]:
nwb_list = fu.load_curated_sessions()


## Data processing

One session (`SESSION_IDX`): signal table, example signals, motion energy on the FIP clock.

### Select a session and build the signal table

`pearsonR` series are dropped; curation adds the region label (`meta['region']`).

In [ ]:
SESSION_IDX = 0
nwb, df_fip, df_trials = fu.select_session(nwb_list, SESSION_IDX)

# Go-cue times on the df_fip clock (first-go-cue-zeroed) drive every peri-event alignment.
cue_col = fu.ALIGN_COL
go_cues = df_trials[fu.ALIGN_COL].dropna().to_numpy()


In [ ]:
meta = fu.build_meta(df_fip)
print(f"{len(meta)} FIP series (pearsonR excluded)")
meta


### Example signals

One trace per signal type: NAc DA (dLight), PL LC-axon calcium (GCaMP), NAc ACh (rAch). `pick_example` prefers a `dff` variant, then the trace with the most finite samples.

In [ ]:
# (label, region substring, plot colour) for each example signal type; the shared default.
EXAMPLE_SPECS = fu.EXAMPLE_SPECS

examples = fu.pick_examples(meta, df_fip)
pd.DataFrame(examples)[["label", "event", "channel", "region", "variant", "color"]]


### Motion energy on the FIP clock

`fu.motion_energy_to_session` reads the aligned ME table (frame times corrected by the video timing QC), divides each value by the exposures it spans, resamples evenly at the camera rate, shifts the clock to the first go cue, and keeps 30 s before the first go cue to 30 s after the last. Refused cameras raise `fu.MotionEnergyRefused`. Frame 0 has no value.

In [ ]:
# This session's row in the aligned ME table: timing action, frames lost, and status.
me_info = fu.load_me_index().query("ses_idx == @nwb.session_id and camera == 'BottomCamera'")
me_info[["session", "action", "status", "frames_lost", "error"]]


In [ ]:
# Put motion energy on the first-go-cue-zeroed session clock (the df_fip axis).
t_me, me, offset = fu.motion_energy_to_session(nwb.session_id, df_trials)


In [ ]:
# Verify before trusting: offset small & positive, ME/FIP time ranges overlap.
print(f"offset = {offset:.3f} s | ME {t_me.min():.1f}..{t_me.max():.1f}s | "
      f"FIP {df_fip['timestamps'].min():.1f}..{df_fip['timestamps'].max():.1f}s")

# Z-scored motion energy + movement onsets (raw, no smoothing); feed the plots.
# su.zscore matches enrich_dfs.zscore_fip's convention (ddof=1), and already_z=True stops
# threshold_onsets from z-scoring a second time.
me_z = su.zscore(me)
# fu.ME_ONSET_KW is the onset rule shared with process_session and men_00 (30 ms above 2.5 SD).
Z_THRESH = fu.ME_ONSET_KW["z_thresh"]
me_onset_times = su.threshold_onsets(t_me, me_z, already_z=True, **fu.ME_ONSET_KW)
print("%d motion-energy onsets at z>%.1f" % (len(me_onset_times), Z_THRESH))


## Data visualization

Green: NAc DA. Blue: PL. Red: NAc ACh.

### Full-session traces

In [ ]:
fig, axes = plt.subplots(len(examples), 1, sharex=True, figsize=(12, 2.2 * len(examples)))
for ax, ex in zip(np.atleast_1d(axes), examples):
    t, y = fu.get_trace(df_fip, ex["event"])
    ax.plot(t, y, color=ex["color"], lw=0.5, alpha=0.85)
    ax.set_ylabel(ex["variant"])
    ax.set_title("%s  -  %s" % (ex["label"], ex["event"]), loc="left", fontsize=10)
np.atleast_1d(axes)[-1].set_xlabel("time from first go cue (s)")
fig.suptitle("Full-session traces - %s" % nwb.session_id)
fig.tight_layout()
plt.show()

### 60-second window with go cues

In [ ]:
WINDOW = (0, 60)  # seconds from first go cue
cues = None
if cue_col is not None:  # cue_col detected in section 6
    cues = df_trials[cue_col]
    cues = cues[(cues >= WINDOW[0]) & (cues <= WINDOW[1])].to_numpy()

fig, axes = plt.subplots(len(examples), 1, sharex=True, figsize=(12, 2.2 * len(examples)))
for ax, ex in zip(np.atleast_1d(axes), examples):
    t, y = fu.get_trace(df_fip, ex["event"])
    m = (t >= WINDOW[0]) & (t <= WINDOW[1])
    ax.plot(t[m], y[m], color=ex["color"], lw=0.9, label=ex["label"])
    if cues is not None:
        for c in cues:
            ax.axvline(c, color="k", lw=0.5, alpha=0.35)
    ax.set_ylabel(ex["variant"])
    ax.legend(loc="upper right", fontsize=8)
np.atleast_1d(axes)[-1].set_xlabel("time from first go cue (s)")
fig.suptitle("First %d s with go cues - %s" % (WINDOW[1], nwb.session_id))
fig.tight_layout()
plt.show()

### Peri-go-cue averages (native dff)

In [ ]:
fig, axes = plt.subplots(1, len(examples), figsize=(4 * len(examples), 3.5))
for ax, ex in zip(np.atleast_1d(axes), examples):
    etr = fu.peri_event(*fu.get_trace(df_fip, ex["event"]), go_cues, t_before=1.0, t_after=3.0)
    pu.plot_etr(ax, etr, ex["color"], lw=1.5, alpha=0.25)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title("%s\n(n=%d cues)" % (ex["label"], etr["event_number"].nunique()), fontsize=9)
    ax.set_xlabel("time from go cue (s)")
np.atleast_1d(axes)[0].set_ylabel("signal (dff)")
fig.suptitle("Peri-go-cue averages")
fig.tight_layout()
plt.show()

### Z-scored motion energy & movement onsets

In [ ]:
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(t_me, me_z, color="purple", lw=0.5, alpha=0.7)
ax.axhline(Z_THRESH, color="k", ls="--", lw=0.8, label="z=%.1f" % Z_THRESH)
ax.plot(me_onset_times, np.full(len(me_onset_times), Z_THRESH), "v", color="k", ms=4,
        label="onsets")
ax.set_xlabel("time from first go cue (s)")
ax.set_ylabel("motion energy (z)")
ax.set_title("Z-scored motion energy - %s" % nwb.session_id)
ax.legend(loc="upper right", fontsize=8)
plt.show()

### FIP aligned to motion-energy onsets

Each signal averaged around motion-energy onsets.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for ex in examples:
    t, y = fu.get_trace(df_fip, ex["event"])
    etr = fu.peri_event(t, su.zscore(y), me_onset_times, t_before=0.5, t_after=1.5, censor=False)
    pu.plot_etr(ax, etr, ex["color"], ex["label"], lw=1.5)
ax.axvline(0, color="k", lw=0.8, ls="--", label="ME onset")
ax.set_xlabel("time from motion-energy onset (s)")
ax.set_ylabel("FIP signal (z)")
ax.set_title("FIP aligned to motion-energy onsets (n=%d)" % len(me_onset_times))
ax.legend(fontsize=8)
plt.show()

### Motion energy aligned to each signal's transients

Motion energy averaged around each signal's transient onsets.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for ex in examples:
    t, y = fu.get_trace(df_fip, ex["event"])
    fip_onsets = su.threshold_onsets(t, y, z_thresh=2.0, refractory=1.0, min_run=3)
    etr = fu.peri_event(t_me, me_z, fip_onsets, t_before=1.0, t_after=1.0, censor=False)
    pu.plot_etr(ax, etr, ex["color"], "%s (n=%d)" % (ex["label"], len(fip_onsets)), lw=1.5)
ax.axvline(0, color="k", lw=0.8, ls="--", label="FIP transient")
ax.set_xlabel("time from FIP transient onset (s)")
ax.set_ylabel("motion energy (z)")
ax.set_title("Motion energy aligned to each signal's transients")
ax.legend(fontsize=8)
plt.show()

### Cross-correlation: motion energy × FIP

Both on a 20 Hz grid over their overlap. Positive lag: FIP later than motion energy.

In [ ]:
FS_XC = 20      # Hz, common resampling grid
MAXLAG = 3.0    # s

fig, ax = plt.subplots(figsize=(7, 4.5))
for ex in examples:
    t, y = fu.get_trace(df_fip, ex["event"])
    finite = np.isfinite(t) & np.isfinite(y)
    t, y = t[finite], y[finite]
    lo = max(t.min(), t_me.min())
    hi = min(t.max(), t_me.max())
    grid = np.arange(lo, hi, 1.0 / FS_XC)
    fip_i = np.interp(grid, t, y)          # FIP on common grid
    mfin = np.isfinite(t_me) & np.isfinite(me_z)
    me_i = np.interp(grid, t_me[mfin], me_z[mfin])  # ME (z) on common grid (interp only)
    lags, c = su.norm_xcorr(fip_i, me_i, fs=FS_XC, maxlag=MAXLAG)
    peak = lags[int(np.argmax(c))]
    ax.plot(lags, c, color=ex["color"], label="%s (peak %+.2fs)" % (ex["label"], peak))
ax.axvline(0, color="k", lw=0.6, ls=":")
ax.set_xlabel("lag (s)   -   positive = motion energy leads FIP")
ax.set_ylabel("normalised cross-correlation")
ax.set_title("Motion energy x FIP cross-correlation")
ax.legend(fontsize=8)
plt.show()

## Multi-session comparison

`fu.process_session` repeats the single-session steps for every curated session with motion energy.

1. FIP aligned to motion-energy onsets.
2. Motion energy aligned to FIP transients.
3. Go-cue-aligned level, 0–2 s (within trial) vs 2–4 s (inter-trial interval).
4. Go-cue responses by reward/failure streak (`num_reward_past`).

Analyses 1–2: session means averaged within animal, then across animals. Analyses 3–4: mean ± SEM over sessions.

In [ ]:
# === Multi-session helpers ===
# The shared single-session pipeline comes from fip_utils; imported by name so the analysis
# cells below read the same as they did when these were defined locally.
from fip_utils import (
    STREAK_BINS,
    get_trace, peri_event,
    iter_region_signals, streak_go_cues, window_mean,
)
from signal_utils import zscore, threshold_onsets

# Notebook-local because they shape this notebook's records; the pooling itself is stats_utils /
# plot_utils. Not plot_fip's fip_psth_multiple_inner_compute: that pools over sessions (here the
# unit is the animal) and needs motion energy injected into every df_fip as a channel.


def session_etr_mean(t, y, event_times, t_before, t_after, censor, censor_times=None):
    """Per-session mean trace across events -> Series indexed by relative time (or None)."""
    if len(np.asarray(event_times)) == 0 or len(np.asarray(t)) == 0:
        return None
    etr = peri_event(t, y, event_times, t_before=t_before, t_after=t_after,
                     censor=censor, censor_times=censor_times)
    if etr is None or etr.empty:
        return None
    return etr.groupby("time")["data"].mean()


def stack_series(series_list):
    """Per-session Series on one ETR time grid -> (time, (n_sessions, n_time) array), or None."""
    series_list = [s for s in series_list if s is not None]
    if not series_list:
        return None
    M = pd.concat(series_list, axis=1)
    return M.index.to_numpy(float), M.to_numpy(float).T


def collect_region_etr(sessions, region_substr, signal_kind, event_kind,
                       t_before, t_after, censor, fip_onset_kw=None, censor_all_go=False):
    """Per-session mean ETR records for one region.
    signal_kind: 'fip' (matched FIP trace, z-scored) or 'me' (motion energy, z).
    event_kind:  'me_onsets' | 'fip_onsets' | 'go_cues'.
    Returns a list of dicts {subject, session, event, series, n_events}."""
    recs = []
    for s, ev in iter_region_signals(sessions, region_substr):
        t_fip, y_fip = get_trace(s["df_fip"], ev)
        ctimes = None
        if event_kind == "me_onsets":
            events = s["me_onsets"]
        elif event_kind == "fip_onsets":
            events = threshold_onsets(t_fip, y_fip, **(fip_onset_kw or {}))
        elif event_kind == "go_cues":
            events = s["go_cues"]
            if censor_all_go:
                ctimes = s["go_cues"]
        else:
            continue
        if signal_kind == "fip":
            t, y = t_fip, zscore(y_fip)
        else:
            t, y = s["t_me"], s["me_z"]
        ser = session_etr_mean(t, y, events, t_before, t_after, censor, censor_times=ctimes)
        if ser is not None:
            recs.append({"subject": s["subject_id"], "session": s["session_id"], "event": ev,
                         "series": ser, "n_events": int(len(np.asarray(events)))})
    return recs


def plot_by_subject(ax, recs, color, label=None, show_subjects=True):
    """Thin line per subject (mean of its sessions) + bold across-subject mean +/- SEM.
    Returns the number of subjects contributing."""
    stacked = stack_series([r["series"] for r in recs])
    if stacked is None:
        return 0
    t, M = stacked
    subjects, per_subject = st.group_means(list(M), [r["subject"] for r in recs])
    pu.plot_mean_sem(ax, t, per_subject, color, label, show_rows=show_subjects, lw=2.2)
    return len(subjects)


In [ ]:
# Run the single-session pipeline over every curated session. Sessions whose ME/video assets
# are missing (or whose trials lack the alignment column) are skipped and logged, not fatal.
sessions = []
skipped = []
for nwb_i in nwb_list:
    try:
        sessions.append(fu.process_session(nwb_i))
    except Exception as e:
        skipped.append((nwb_i.session_id, "%s: %s" % (type(e).__name__, e)))

print("Usable ME in %d / %d curated sessions (%d skipped)."
      % (len(sessions), len(nwb_list), len(skipped)))
subjects = sorted(set(s["subject_id"] for s in sessions))
print("Subjects with data: %d -> %s" % (len(subjects), subjects))
if skipped:
    print("\nSkipped (first 25):")
    for sid, why in skipped[:25]:
        print("  %-22s %s" % (sid, why))

### Analysis 1 — FIP aligned to motion-energy onsets

One panel per region; thin lines are animals.

In [ ]:
# Analysis 1 — FIP (z) aligned to motion-energy onsets, across sessions.
# Per session, align each region's FIP trace to that session's ME onsets and average; then a
# thin line per subject (mean of its sessions) and a bold across-subject mean ± SEM. One panel
# per region.
res_fip_from_me = []
fig, axes = plt.subplots(1, len(EXAMPLE_SPECS), figsize=(4 * len(EXAMPLE_SPECS), 3.6),
                         sharex=True)
for ax, (label, substr, color) in zip(np.atleast_1d(axes), EXAMPLE_SPECS):
    recs = collect_region_etr(sessions, substr, signal_kind="fip", event_kind="me_onsets",
                              t_before=0.5, t_after=1.5, censor=False)
    n_subj = plot_by_subject(ax, recs, color, label)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title("%s\n%d sessions, %d subjects"
                 % (label, len(set(r["session"] for r in recs)), n_subj), fontsize=9)
    ax.set_xlabel("time from ME onset (s)")
    for r in recs:
        res_fip_from_me.append({"region": substr, "subject": r["subject"],
                                "session": r["session"], "condition": "all",
                                "amp_0_1s": window_mean(r["series"], 0.0, 1.0),
                                "n_events": r["n_events"]})
np.atleast_1d(axes)[0].set_ylabel("FIP (z)")
fig.suptitle("Analysis 1 — FIP aligned to motion-energy onsets (across sessions)")
fig.tight_layout()
plt.show()
res_fip_from_me = pd.DataFrame(res_fip_from_me)
print("res_fip_from_me:", res_fip_from_me.shape)

### Analysis 2 — Motion energy aligned to FIP transients

Transients: `su.threshold_onsets` on each region's trace (2 SD, 1 s refractory). Thin lines are animals.

In [ ]:
# Analysis 2 — motion energy aligned to each region's FIP transients (reverse direction).
# Per session, detect FIP transients (threshold_onsets) on the region's trace and align ME (z)
# to them; average per session, then mean ± SEM across subjects/sessions. One panel per region.
res_me_from_fip = []
fig, axes = plt.subplots(1, len(EXAMPLE_SPECS), figsize=(4 * len(EXAMPLE_SPECS), 3.6),
                         sharex=True)
for ax, (label, substr, color) in zip(np.atleast_1d(axes), EXAMPLE_SPECS):
    recs = collect_region_etr(sessions, substr, signal_kind="me", event_kind="fip_onsets",
                              t_before=1.0, t_after=1.0, censor=False,
                              fip_onset_kw=dict(z_thresh=2.0, refractory=1.0, min_run=3))
    n_subj = plot_by_subject(ax, recs, color, label)
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title("%s\n%d sessions, %d subjects"
                 % (label, len(set(r["session"] for r in recs)), n_subj), fontsize=9)
    ax.set_xlabel("time from FIP transient (s)")
    for r in recs:
        res_me_from_fip.append({"region": substr, "subject": r["subject"],
                                "session": r["session"], "condition": "all",
                                "amp_pre": window_mean(r["series"], -0.5, 0.0),
                                "amp_post": window_mean(r["series"], 0.0, 0.5),
                                "n_events": r["n_events"]})
np.atleast_1d(axes)[0].set_ylabel("motion energy (z)")
fig.suptitle("Analysis 2 — Motion energy aligned to FIP transients (across sessions)")
fig.tight_layout()
plt.show()
res_me_from_fip = pd.DataFrame(res_me_from_fip)
print("res_me_from_fip:", res_me_from_fip.shape)

### Analysis 3 — Within trial vs inter-trial interval

Go-cue-aligned 0–4 s, censored at the next go cue. Mean over 0–2 s vs 2–4 s per signal.

In [ ]:
# Analysis 3 — within-trial (0–2 s) vs ITI (2–4 s) relative to go cue.
# Each signal is averaged peri-go-cue over a 0–4 s window (censored against all go cues), then
# split into a within-trial window (0–2 s) and an ITI window (2–4 s). Left: across-session
# grand-mean traces with the two windows shaded. Right: within-trial vs ITI means (±SEM across
# sessions) per signal.
res_trial_iti = []
agg_for_plot = {}  # signal label -> (time, per-session traces, color)

for label, substr, color in EXAMPLE_SPECS:  # FIP regions
    recs = collect_region_etr(sessions, substr, signal_kind="fip", event_kind="go_cues",
                              t_before=0.0, t_after=4.0, censor=True, censor_all_go=True)
    a = stack_series([r["series"] for r in recs])
    if a is not None:
        agg_for_plot[label] = (a[0], a[1], color)
    for r in recs:
        res_trial_iti.append({"signal": label, "region": substr, "subject": r["subject"],
                              "session": r["session"],
                              "within_trial": window_mean(r["series"], 0.0, 2.0),
                              "iti": window_mean(r["series"], 2.0, 4.0)})

me_sers = []  # motion energy aligned to go cues
for s in sessions:
    ser = session_etr_mean(s["t_me"], s["me_z"], s["go_cues"], 0.0, 4.0, True,
                           censor_times=s["go_cues"])
    if ser is None:
        continue
    me_sers.append(ser)
    res_trial_iti.append({"signal": "motion energy", "region": "ME", "subject": s["subject_id"],
                          "session": s["session_id"],
                          "within_trial": window_mean(ser, 0.0, 2.0),
                          "iti": window_mean(ser, 2.0, 4.0)})
a = stack_series(me_sers)
if a is not None:
    agg_for_plot["motion energy"] = (a[0], a[1], "purple")
res_trial_iti = pd.DataFrame(res_trial_iti)

fig, (axL, axR) = plt.subplots(1, 2, figsize=(13, 4))
for label, (t, M, color) in agg_for_plot.items():
    pu.plot_mean_sem(axL, t, M, color, label, lw=1.8, alpha=0.15)
axL.axvspan(0, 2, color="green", alpha=0.06)
axL.axvspan(2, 4, color="gray", alpha=0.08)
axL.axvline(0, color="k", lw=0.8, ls="--")
axL.set_xlabel("time from go cue (s)")
axL.set_ylabel("signal (z)")
axL.set_title("Go-cue aligned (0–2 s within-trial | 2–4 s ITI)")
axL.legend(fontsize=7)

sig_order = [l for l in ["motion energy"] + [e[0] for e in EXAMPLE_SPECS]
             if l in set(res_trial_iti["signal"])]
x = np.arange(len(sig_order))
w = 0.38
wt_m = [res_trial_iti.loc[res_trial_iti.signal == s, "within_trial"].mean() for s in sig_order]
wt_e = [res_trial_iti.loc[res_trial_iti.signal == s, "within_trial"].sem() for s in sig_order]
it_m = [res_trial_iti.loc[res_trial_iti.signal == s, "iti"].mean() for s in sig_order]
it_e = [res_trial_iti.loc[res_trial_iti.signal == s, "iti"].sem() for s in sig_order]
axR.bar(x - w / 2, wt_m, w, yerr=wt_e, label="within-trial (0–2 s)", color="green", alpha=0.7)
axR.bar(x + w / 2, it_m, w, yerr=it_e, label="ITI (2–4 s)", color="gray", alpha=0.7)
axR.axhline(0, color="k", lw=0.6)
axR.set_xticks(x)
axR.set_xticklabels(sig_order, rotation=20, ha="right", fontsize=8)
axR.set_ylabel("mean signal (z)")
axR.set_title("Within-trial vs ITI (mean ± SEM across sessions)")
axR.legend(fontsize=7)
fig.suptitle("Analysis 3 — Within-trial vs ITI relative to go cue")
fig.tight_layout()
plt.show()
print("res_trial_iti:", res_trial_iti.shape)

### Analysis 4 — Reward/failure streaks

Go cues binned by `num_reward_past` (consecutive rewards positive, failures negative; Rachel's `enrich_df_trials`). Go-cue-aligned FIP per bin.

In [ ]:
# Analysis 4 — FIP peri-go-cue averages split by reward/failure streak (num_reward_past).
# Go cues are binned by how many consecutive rewards (+) or failures (-) preceded the trial;
# each bin's go cues drive a peri-event average, censored against ALL go cues so neighbouring
# trials don't bleed in. One panel per region; one line per streak bin (across-session mean).
res_streak = []
fig, axes = plt.subplots(1, len(EXAMPLE_SPECS), figsize=(4.6 * len(EXAMPLE_SPECS), 3.8),
                         sharex=True)
for ax, (label, substr, color) in zip(np.atleast_1d(axes), EXAMPLE_SPECS):
    for bin_label, bin_fn, bcolor in STREAK_BINS:
        sers = []
        for s, ev in iter_region_signals(sessions, substr):
            ev_times = streak_go_cues(s["df_trials"], bin_fn)
            if ev_times is None or len(ev_times) == 0:
                continue
            t, y = get_trace(s["df_fip"], ev)
            ser = session_etr_mean(t, zscore(y), ev_times, 1.0, 3.0, True,
                                   censor_times=s["go_cues"])
            if ser is not None:
                sers.append(ser)
                res_streak.append({"region": substr, "subject": s["subject_id"],
                                   "session": s["session_id"], "condition": bin_label,
                                   "n_events": int(len(ev_times)),
                                   "amp_0_1s": window_mean(ser, 0.0, 1.0)})
        stacked = stack_series(sers)
        if stacked is None:
            continue
        t, M = stacked
        ax.plot(t, st.mean_sem(M)[0], color=bcolor, lw=1.6, label="%s (n=%d)" % (bin_label, len(M)))
    ax.axvline(0, color="k", lw=0.8, ls="--")
    ax.set_title(label, fontsize=9)
    ax.set_xlabel("time from go cue (s)")
    ax.legend(fontsize=6, ncol=2)
np.atleast_1d(axes)[0].set_ylabel("FIP (z)")
fig.suptitle("Analysis 4 — FIP peri-go-cue by reward/failure streak (across sessions)")
fig.tight_layout()
plt.show()
res_streak = pd.DataFrame(res_streak)
print("res_streak:", res_streak.shape)

### Cross-session results table

Scalar summaries of Analyses 1–4 in one table, `df_results`.

In [ ]:
# Pool the per-analysis scalar summaries into one tidy table (in-memory; session is the
# sampling unit). Schemas differ per analysis, so columns are unioned (NaN where N/A).
_tables = {"fip_from_me": globals().get("res_fip_from_me"),
           "me_from_fip": globals().get("res_me_from_fip"),
           "trial_vs_iti": globals().get("res_trial_iti"),
           "streak": globals().get("res_streak")}
parts = []
for _name, _obj in _tables.items():
    if isinstance(_obj, pd.DataFrame) and not _obj.empty:
        print("%-13s %s" % (_name, tuple(_obj.shape)))
        parts.append(_obj.assign(table=_name))
df_results = pd.concat(parts, ignore_index=True, sort=False) if parts else pd.DataFrame()
print("df_results:", df_results.shape)
df_results.head()

### (Optional) Motion energy as a FIP channel

`BUILD_NWB_LIST_ME = True` adds motion energy to each `df_fip` as event `"ME"`, for `plot_fip`'s PSTH functions.

In [ ]:
# Optional: inject ME as a FIP pseudo-channel (event='ME') so the upstream multi-session
# PSTH machinery (aind_dynamic_foraging_basic_analysis.plot.plot_fip) can treat motion
# energy exactly like a FIP channel — it selects signals via df_fip['event'] == channel, so
# 'ME' becomes usable by plot_fip_psth_compare_alignments / _compare_channels /
# fip_psth_multiple_inner_compute (incl. their hierarchical-bootstrap error bars).
# Off by default: it appends ME rows to every session's df_fip (memory). The analyses above
# use the lighter in-notebook SEM-across-sessions path and do not need this.
BUILD_NWB_LIST_ME = False

if BUILD_NWB_LIST_ME:
    nwb_list_me = []
    for s in sessions:
        nwb_i = s["nwb"]
        nwb_i.df_fip = fu.attach_me_to_df_fip(nwb_i.df_fip, s["t_me"], s["me"], nwb_i.session_id)
        nwb_i.df_trials = s["df_trials"]
        nwb_list_me.append(nwb_i)
    print("nwb_list_me ready (%d sessions); 'ME' is now a channel in df_fip." % len(nwb_list_me))
    print("ME is stored RAW; z-score it the same way as any FIP channel before plotting:")
    print("     from aind_dynamic_foraging_data_utils import enrich_dfs")
    print("     nwb_i.df_fip = enrich_dfs.zscore_fip(nwb_i.df_fip)   # adds data_z")
    print("e.g. from aind_dynamic_foraging_basic_analysis.plot import plot_fip")
    print("     plot_fip.fip_psth_multiple_inner_compute(nwb_list_me, me_onsets_list, "
          "channel='ME', average=True, tw=[-0.5, 1.5], censor=False, data_column='data_z')")
else:
    print("BUILD_NWB_LIST_ME=False — using the in-notebook SEM-across-sessions path.")